# FINAL PROTOCOL: 4-Channel Architecture with Learned ROI Cropping
- 4-Channel
- Box Regressor
- Visualization
- TTA Submission


## Imports

In [ ]:
import os
import csv
import random
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from sklearn.utils.class_weight import compute_class_weight

import tifffile as tiff 
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from torchvision.transforms import RandAugment
from torch.cuda.amp import autocast, GradScaler # Mixed Precision for speed

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

## Configuration

In [ ]:
class Config:
    SEED = 42
    
    # TRAIN PATHS (Cleaned Data)
    # Ensure these point to the cleaned dataset we generated earlier
    CLEAN_IMG_DIR = '/kaggle/input/clean-ds/final_clean_dataset/images'
    CLEAN_MASK_DIR = '/kaggle/input/clean-ds/final_clean_dataset/masks'
    CLEAN_LABELS  = '/kaggle/input/clean-ds/final_clean_dataset/train_labels.csv'
    
    # TIFF PATHS
    TIFF_DIR = '/kaggle/input/train-tiff-ds/' 
    
    # TEST PATHS + TIFF
    TEST_DIR = '/kaggle/input/grumpy-doctogres/test_data'
    TEST_TIFF_DIR = '/kaggle/input/test-tiff-ds/'
    
    # HYPERPARAMS
    IMG_SIZE_BOX = 224   # Low res for finding the tissue (Regressor)
    IMG_SIZE_CLS = 512   # High res for classifying details (Classifier)
    BATCH_SIZE = 8       
    NUM_CLASSES = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    EPOCHS_BOX = 200     # Regressor converges fast (early stopping used)
    EPOCHS_CLS = 500     # Classifier needs more time (early stopping used)
    LR = 1e-4            # Lower LR for fine-tuning
    ACCUM_STEPS = 4      # Effective Batch Size = 8 * 4 = 32
    WD = 1e-3            # Weight Decay (Increased for stronger regularization)
    DROPOUT = 0.4        # Dropout

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(Config.SEED)
print(f"Operational Device: {Config.DEVICE}")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
            verbose (bool): If True, prints a message for each validation loss improvement.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
            path (str): Path for the checkpoint to be saved to.
            trace_func (function): trace print function.
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_metric, model):
        # Monitor F1: Higher is better
        # If the metric is lower the better, just pass the negated score
        score = val_metric

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'   EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
            self.counter = 0

    def save_checkpoint(self, val_metric, model):
        '''Saves model when validation F1 increases.'''
        if self.verbose:
            abs_score = abs(self.best_score)
            self.trace_func(f'   Better score detected ({abs_score:.4f}). Saving model...')
        torch.save(model.state_dict(), self.path)

## PHASE 1: The Box Regressor

In [ ]:
# Dataset for Regression
class BoxDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, transform=None, cache_data=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.cache_data = cache_data
        self.cache = []

        # Caching to speedup
        if self.cache_data:
            print(f"   Pre-loading {len(df)} images into RAM for faster training...")
            for idx in range(len(df)):
                self.cache.append(self.load_item(idx))
            print("   ✅ Caching Complete.")
        
    def __len__(self):
        return len(self.df)

    def get_normalized_bbox(self, mask_np, width, height):
        rows = np.any(mask_np, axis=1)
        cols = np.any(mask_np, axis=0)
        if not np.any(rows) or not np.any(cols): 
            return torch.tensor([0.0, 0.0, 1.0, 1.0], dtype=torch.float32) # Full image fallback
        
        ymin, ymax = np.where(rows)[0][[0, -1]]
        xmin, xmax = np.where(cols)[0][[0, -1]]
        
        # Add padding (optional)
        pad = 20
        ymin = max(0, ymin - pad)
        xmin = max(0, xmin - pad)
        ymax = min(height, ymax + pad)
        xmax = min(width, xmax + pad)
        
        # Normalize coordinates [0, 1]
        return torch.tensor([xmin/width, ymin/height, xmax/width, ymax/height], dtype=torch.float32)

    def load_item(self, idx):
        row = self.df.iloc[idx]
        fname = row['sample_index']
        
        # Load Image
        img_path = os.path.join(self.img_dir, fname)
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        
        # Load Mask to get Ground Truth Box
        mask_name = fname.replace("image_", "mask_")
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        if os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert("L"))
            box = self.get_normalized_bbox(mask, w, h)
        else:
            box = torch.tensor([0.0, 0.0, 1.0, 1.0], dtype=torch.float32)
            
        return image, box

    
    def __getitem__(self, idx):
        if self.cache_data:
            image, box = self.cache[idx]  # Faster
        else:
            image, box = self.load_item(idx)
            
        if self.transform:
            image = self.transform(image)
            
        return image, box

In [ ]:
# Regressor Model
class BoxRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        # EfficientNetB0 is perfect for this: lightweight and fast
        self.backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        
        # Replace Classifier Head
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, 4), # Predicts x1, y1, x2, y2
            nn.Sigmoid() # Forces output to be [0, 1]
        )
        
    def forward(self, x):
        return self.backbone(x)

In [ ]:
# Training Function for Regressor
def train_regressor(df):
    print("\nPHASE 1: Training The Box Regressor...")

    # Split Data for Validation
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=Config.SEED)
    
    # Transform (Resize to 224 is enough for localization)
    transform = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE_BOX, Config.IMG_SIZE_BOX)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_ds = BoxDataset(train_df, Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, transform=transform, cache_data=True)
    val_ds   = BoxDataset(val_df,   Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, transform=transform, cache_data=True)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE * 2, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE * 2, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    
    model = BoxRegressor().to(Config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # criterion = nn.MSELoss() # Mean Squared Error for regression
    criterion = nn.SmoothL1Loss(beta=1.0) # NEW: Robust to outliers
    
    scaler = torch.cuda.amp.GradScaler() # AMP

    # Initialize EARLY STOPPING
    # - Monitor Validation Loss (MSE)
    # - Patience=15 because regressors can be jittery.
    early_stopping = EarlyStopping(patience=30, path="rgss_best_model.pth")

    # History Dictionary
    history = {
        "train_loss": [], 
        "val_loss": [], 
        "val_spearman": []
    }
    
    for epoch in range(Config.EPOCHS_BOX):
        # --- TRAIN ---
        model.train()
        train_loss = 0
        for imgs, boxes in tqdm(train_loader, desc=f"Regressor Epoch {epoch+1}", leave=False):
            imgs, boxes = imgs.to(Config.DEVICE), boxes.to(Config.DEVICE)
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                preds = model(imgs)
                loss = criterion(preds, boxes)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * imgs.size(0)
        
        train_loss_epoch = train_loss / len(train_loader.dataset)

        # --- VALIDATE ---
        model.eval()
        val_loss = 0
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for imgs, boxes in val_loader:
                imgs, boxes = imgs.to(Config.DEVICE), boxes.to(Config.DEVICE)
                preds = model(imgs)
                loss = criterion(preds, boxes)
                val_loss += loss.item() * imgs.size(0)
                
                all_preds.extend(preds.cpu().numpy().flatten())
                all_targets.extend(boxes.cpu().numpy().flatten())
        
        val_loss_epoch = val_loss / len(val_loader.dataset)
        
        # Calculate Spearman (Flattened coords correlation)
        val_spearman, _ = spearmanr(all_targets, all_preds)

        # Record History
        history["train_loss"].append(train_loss_epoch)
        history["val_loss"].append(val_loss_epoch)
        history["val_spearman"].append(val_spearman)
        
        print(f"Epoch {epoch+1:4d}/{Config.EPOCHS_BOX} | Train SmoothL1 {train_loss_epoch:.4f} | Val SmoothL1 {val_loss_epoch:.4f} | Val Spearman {val_spearman:.4f}")        

        # Check EARLY STOPPING
        # Pass negative version because we want to minimize error.
        early_stopping(-val_loss_epoch, model)
        # early_stopping(val_spearman, model)

        if early_stopping.early_stop:
            print(" Early stopping triggered!")
            break
            
    # Load Best Weights
    model.load_state_dict(torch.load("rgss_best_model.pth"))
    print("✅ Regressor Trained & Saved.")
    return model, history

## PHASE 2: The 4-Channel Classifier

In [ ]:
# 4-Channel Dataset
class ClassifierDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, tiff_dir, transform=None, mode='train', cache_data=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.tiff_dir = tiff_dir
        self.transform = transform
        self.mode = mode 
        self.le = LabelEncoder()
        self.cache_data = cache_data
        self.cache = []
        
        # Fit on provided df 
        if 'label' in self.df.columns:
            self.df['encoded_label'] = self.le.fit_transform(self.df['label'])

        # Pre-loading sequence for faster training
        if self.cache_data:
            print(f"   Pre-loading {len(df)} 4-Channel images into RAM...")
            for idx in tqdm(range(len(df)), desc="Caching Classifier"):
                self.cache.append(self.load_item(idx))
            print("   ✅ Caching Complete.")

        # RandAugment (RGB ONLY)
        self.randaug = RandAugment(num_ops=2, magnitude=7)

    def __len__(self):
        return len(self.df)
    
    def get_bbox_from_mask(self, mask_np):
        rows = np.any(mask_np, axis=1)
        cols = np.any(mask_np, axis=0)
        if not np.any(rows) or not np.any(cols): return None
        ymin, ymax = np.where(rows)[0][[0, -1]]
        xmin, xmax = np.where(cols)[0][[0, -1]]
        return xmin, ymin, xmax, ymax

    def load_item(self, idx):
        row = self.df.iloc[idx]
        fname = row['sample_index']
        
        # 1. Load RGB
        img_path = os.path.join(self.img_dir, fname)
        rgb = Image.open(img_path).convert("RGB")
        
        # 2. Load TIFF (Fallback logic)
        tiff_name = fname.replace(".png", ".tiff")
        tiff_path = os.path.join(self.tiff_dir, tiff_name)
        
        if os.path.exists(tiff_path):
            try:
                tif_data = tiff.imread(tiff_path)
                tif_norm = (tif_data - tif_data.min()) / (tif_data.max() - tif_data.min() + 1e-6)
                extra_channel = Image.fromarray((tif_norm * 255).astype(np.uint8)).convert("L")
            except: extra_channel = Image.new('L', rgb.size, 0)
        else: extra_channel = Image.new('L', rgb.size, 0)

        # 3. Smart Crop (Train only - using Masks)
        # Note: At test time, we will use the regressor Model to crop!
        if self.mode == 'train':
            mask_name = fname.replace("image_", "mask_")
            mask_path = os.path.join(self.mask_dir, mask_name)
            if os.path.exists(mask_path):
                mask = Image.open(mask_path).convert("L")
                bbox = self.get_bbox_from_mask(np.array(mask))
                if bbox:
                    xmin, ymin, xmax, ymax = bbox
                    pad = 30

                    # --- Improved cropping to avoid tiny crops or excessive trimming ---
                    xmin, ymin = max(0, xmin - pad), max(0, ymin - pad)
                    xmax, ymax = min(rgb.width,  xmax + pad), min(rgb.height, ymax + pad)
                    
                    # Ensure minimum crop size
                    min_size = 96
                    if (xmax - xmin) < min_size:
                        delta = (min_size - (xmax - xmin)) // 2
                        xmin = max(0, xmin - delta)
                        xmax = min(rgb.width, xmax + delta)
                    
                    if (ymax - ymin) < min_size:
                        delta = (min_size - (ymax - ymin)) // 2
                        ymin = max(0, ymin - delta)
                        ymax = min(rgb.height, ymax + delta)
                    
                    # Expand crop by a factor to capture context
                    expand = 1.20
                    cx = (xmin + xmax) // 2
                    cy = (ymin + ymax) // 2
                    w2 = int((xmax - xmin) * expand / 2)
                    h2 = int((ymax - ymin) * expand / 2)
                    
                    xmin = max(0, cx - w2)
                    xmax = min(rgb.width,  cx + w2)
                    ymin = max(0, cy - h2)
                    ymax = min(rgb.height, cy + h2)
                    
                    rgb = rgb.crop((xmin, ymin, xmax, ymax))
                    extra_channel = extra_channel.crop((xmin, ymin, xmax, ymax))


        # 4. Resize to High Res (512)
        rgb = rgb.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
        extra_channel = extra_channel.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
        
        # 5. Stack (but normalize later)
        rgb_t = TF.to_tensor(rgb)
        extra_t = TF.to_tensor(extra_channel)
        full_img = torch.cat([rgb_t, extra_t], dim=0)  # shape: (4, H, W)
            
        label = row['encoded_label'] if 'encoded_label' in row else -1
        
        return full_img, label


    def __getitem__(self, idx):
        # 1. Fetch Base Tensor (Instant)
        if self.cache_data:
            full_img, label = self.cache[idx]
        else:
            full_img, label = self.load_item(idx)
            
        # 2. Apply Augmentation (Manual for 4-channel - dynamically per epoch)
        if self.mode == 'train':
            
            # --- Geometric Augmentations ---
            # RandomResizedCrop
            if random.random() > 0.3:
                i, j, h, w = transforms.RandomResizedCrop.get_params(
                    full_img, scale=(0.8, 1.0), ratio=(0.9, 1.1)
                )
                full_img = TF.resized_crop(full_img, i, j, h, w, (Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS))

            # flips
            if random.random() > 0.5: full_img = TF.hflip(full_img)
            if random.random() > 0.5: full_img = TF.vflip(full_img)

            # rotations
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                full_img = TF.rotate(full_img, angle)

            # Extract RGB + TIFF
            rgb = full_img[:3, :, :]
            tif = full_img[3:, :, :]
            
            # ---------- RAND-AUGMENT (RGB ONLY) ----------
            # Convert RGB tensor (float32) → PIL uint8
            rgb_pil = transforms.functional.to_pil_image(rgb)
    
            # Apply RandAugment policy
            rgb_pil = self.randaug(rgb_pil)
    
            # Convert back to tensor
            rgb_aug = transforms.functional.to_tensor(rgb_pil)
    
            # Replace RGB in the original 4-channel tensor
            full_img = torch.cat([rgb_aug, tif], dim=0)

        # 3. NORMALIZATION (APPLY LAST)
        mean = [0.485, 0.456, 0.406, 0.5]
        std  = [0.229, 0.224, 0.225, 0.5]
        full_img = TF.normalize(full_img, mean=mean, std=std)
        
        return full_img, label

In [ ]:
# Classifier Model
def get_classifier_model(num_classes):
    print("Instantiating Classifier (ConvNeXt-Tiny 4-Channel)...")
    # ConvNeXt Tiny is roughly equal to ResNet50 but much better features
    model = models.convnext_tiny(weights='IMAGENET1K_V1')

    # 1. Modify First Layer (3 -> 4 channels)
    # ConvNeXt starts with: model.features[0][0] which is Conv2d
    old_conv = model.features[0][0]
    new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=old_conv.kernel_size, 
                         stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None)
    
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:4] = torch.mean(old_conv.weight, dim=1, keepdim=True)
        if old_conv.bias is not None:
            new_conv.bias = nn.Parameter(old_conv.bias.clone())

    model.features[0][0] = new_conv

    # 2. Modify Head (Classifier)
    # ConvNeXt head structure: model.classifier[2] is the Linear layer
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    
    return model.to(Config.DEVICE)

In [ ]:
# Classifier Training Loop
def train_classifier(df):
    print("\nPHASE 2: Training The Classifier...")

    # Setup & Split
    train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=Config.SEED)
    
    train_ds = ClassifierDataset(train_df, Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TIFF_DIR, mode='train', cache_data=True)
    val_ds   = ClassifierDataset(val_df,   Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TIFF_DIR, mode='val', cache_data=True)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    # Calculate class weights (Fixes the "Triple Negative = 0.00" issue)
    print("   Computing Class Weights...")
    all_labels = train_df['encoded_label'].values
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(Config.DEVICE)
    print(f"   Weights: {class_weights.cpu().numpy()}")
    
    model = get_classifier_model(Config.NUM_CLASSES)

    # Pass weights to Loss
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WD)
    scaler = torch.cuda.amp.GradScaler()

    # Initialize EARLY STOPPING
    early_stopping = EarlyStopping(patience=30, path="cls_best_model.pth")

    # History Dictionary
    history = {
        'train_loss':[], 
        'train_acc':[], 
        'val_loss':[], 
        'val_acc':[], 
        'val_f1':[]
    }
    
    for epoch in range(Config.EPOCHS_CLS):
        # --- TRAIN ---
        model.train()
        t_loss_sum, t_corr, t_total = 0, 0, 0

        optimizer.zero_grad()  # Reset before accumulation starts
        
        for i, (imgs, lbls) in enumerate(tqdm(train_loader, desc=f"Classifier Epoch {epoch+1}", leave=False)):
            imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
            
            with torch.cuda.amp.autocast():
                out = model(imgs)
                loss = criterion(out, lbls)
                loss = loss / Config.ACCUM_STEPS # Normalize loss
            
            scaler.scale(loss).backward()
            
            # Step only after accumulation
            if (i + 1) % Config.ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            # Stats (undo normalization for logging)
            t_loss_sum += loss.item() * Config.ACCUM_STEPS * imgs.size(0)
            _, preds = torch.max(out, 1)
            t_corr += (preds == lbls).sum().item()
            t_total += lbls.size(0)
            
        t_loss = t_loss_sum / t_total
        t_acc = t_corr / t_total
            
        # --- VALIDATION ---
        model.eval()
        v_loss_sum, v_corr, v_total = 0, 0, 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                out = model(imgs)
                loss = criterion(out, lbls)

                v_loss_sum += loss.item() * imgs.size(0)
                _, preds = torch.max(out, 1)
                v_corr += (preds == lbls).sum().item()
                v_total += lbls.size(0)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())
                
        # Calculate metrics
        v_loss = v_loss_sum / v_total
        v_acc = v_corr / v_total
        v_f1 = f1_score(all_labels, all_preds, average='macro')
        
        # History
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        history['val_f1'].append(v_f1)
        
        print(f"Epoch {epoch+1:4d}/{Config.EPOCHS_CLS} | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.4f} | Val F1: {v_f1:.4f}")        

        # --- CHECK EARLY STOPPING ---
        early_stopping(v_f1, model)
        
        if early_stopping.early_stop:
            print(" Early stopping triggered!")
            break
            
    # Load the best model saved by EarlyStopping
    model.load_state_dict(torch.load("cls_best_model.pth"))

    return model, train_ds.le, history, early_stopping.best_score, val_loader # Return everything needed for plots/submission

## PHASE 3: Smart Inference (Regressor + Classifier + TTA)

In [ ]:
def smart_inference(regressor, classifier, le, test_dir, tiff_dir, f1_score=0.0):
    print("\nPHASE 3: Smart Inference (Regressor -> Crop -> Classifier)...")
    
    regressor.eval()
    classifier.eval()
    
    # Regressor Transform
    regressor_trans = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE_BOX, Config.IMG_SIZE_BOX)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Normalization for Classifier
    mean_4ch = [0.485, 0.456, 0.406, 0.5]
    std_4ch  = [0.229, 0.224, 0.225, 0.5]
    
    test_files = sorted([f for f in os.listdir(test_dir) if (f.endswith('.png'))&(f.startswith('img_'))])
    results = []
    
    with torch.no_grad():
        for fname in tqdm(test_files, desc="Processing"):
            img_path = os.path.join(test_dir, fname)
            
            # A. LOAD & SNIPE
            rgb_orig = Image.open(img_path).convert("RGB")
            w, h = rgb_orig.size
            
            # Predict Box
            regressor_input = regressor_trans(rgb_orig).unsqueeze(0).to(Config.DEVICE)
            box_pred = regressor(regressor_input).cpu().numpy()[0] # [x1, y1, x2, y2] normalized
            
            # Denormalize Coords & Sanitize Coords
            x1 = int(box_pred[0] * w)
            y1 = int(box_pred[1] * h)
            x2 = int(box_pred[2] * w)
            y2 = int(box_pred[3] * h)

            # Safety Check
            # Fix: ensure x1 < x2 and y1 < y2, and clamp to image bounds
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(x1 + 1, min(x2, w))
            y2 = max(y1 + 1, min(y2, h))
            
            # # If the predicted box is tiny (garbage prediction), use full image
            # if (x2 - x1) < 10 or (y2 - y1) < 10:
            #     x1, y1, x2, y2 = 0, 0, w, h

            # Fallback for unreasonable crops
            min_size = min(w, h) * 0.10  # 10% of image
            if (x2 - x1) < min_size or (y2 - y1) < min_size:
                x1, y1, x2, y2 = 0, 0, w, h
            
            # Expand predicted box for more context
            expand = 1.25
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            bw = int((x2 - x1) * expand / 2)
            bh = int((y2 - y1) * expand / 2)
            
            x1 = max(0, cx - bw)
            x2 = min(w, cx + bw)
            y1 = max(0, cy - bh)
            y2 = min(h, cy + bh)

                
            # Crop RGB
            rgb_crop = rgb_orig.crop((x1, y1, x2, y2))
            
            # B. LOAD TIFF & CROP
            tiff_name = fname.replace(".png", ".tiff")
            tiff_path = os.path.join(tiff_dir, tiff_name)
            if os.path.exists(tiff_path):
                try:
                    t = tiff.imread(tiff_path)
                    t_norm = (t - t.min()) / (t.max() - t.min() + 1e-6)
                    tif_img = Image.fromarray((t_norm*255).astype(np.uint8)).convert("L")
                    tif_crop = tif_img.crop((x1, y1, x2, y2))
                except: tif_crop = Image.new('L', rgb_crop.size, 0)
            else:
                tif_crop = Image.new('L', rgb_crop.size, 0)
                
            # C. CLASSIFIER PREP
            rgb_final = rgb_crop.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
            tif_final = tif_crop.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
            
            # Stack
            rgb_t = TF.to_tensor(rgb_final)
            tif_t = TF.to_tensor(tif_final)
            full_t = torch.cat([rgb_t, tif_t], dim=0)
            full_t = TF.normalize(full_t, mean=mean_4ch, std=std_4ch)
            
            # D. TTA (Flip/Rotate)
            views = [full_t]
            views.append(TF.hflip(full_t))
            views.append(TF.vflip(full_t))
            views.append(TF.rotate(full_t, 90))
            
            batch = torch.stack(views).to(Config.DEVICE)
            
            # E. PREDICT
            out = classifier(batch)
            prob = F.softmax(out, dim=1).mean(dim=0) # Average TTA
            pred_idx = prob.argmax().item()
            
            results.append([fname, le.inverse_transform([pred_idx])[0]])
            
    # Save
    # Define Filename based on score
    f1_string = f"{f1_score:.4f}".replace(".", "")
    submission_filename = f"submission_f1_{f1_string}.csv"
    with open(submission_filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['sample_index', 'label'])
        writer.writerows(results)
    print(f"✅ Submission Saved: {submission_filename}")

## Visualization Methods

In [ ]:
# Plot REGRESSOR History
def plot_history_box(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(history["train_loss"], label="Train", alpha=0.5)
    ax1.plot(history["val_loss"], label="Val", alpha=0.9)
    ax1.set_title("Box regression MSE loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history["val_spearman"])
    ax2.set_title("Spearman rho on validation")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Rho")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# Plot CLASSIFIER History
def plot_history_cls(history, title_suffix=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(history["train_loss"], label="Train", alpha=0.5)
    ax1.plot(history["val_loss"], label="Val", alpha=0.9)
    ax1.set_title(f"Cross entropy loss{title_suffix}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", alpha=0.5)
    ax2.plot(history["val_acc"], label="Val", alpha=0.9)
    ax2.set_title(f"Accuracy{title_suffix}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # F1 plot for this specific history
    plt.figure(figsize=(7, 5))
    plt.plot(history["val_f1"])
    plt.title(f"Validation F1 (macro){title_suffix}")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Plot Confusion Matrix
def plot_confusion_matrix_model(model, loader, class_names):
    print("📊 Generating Confusion Matrix...")
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for img, lbl in loader:
            img = img.to(Config.DEVICE)
            lbl = lbl.to(Config.DEVICE)
            out = model(img)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbl.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Validation confusion matrix")
    plt.tight_layout()
    plt.show()

    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
def visualize_test_predictions(images, cls_probs, box_preds, class_names, max_images=10):
    """
    Visualizes 
    """
    print("📊 Visualizing Test Predictions...")
    num_img = len(images)
    n = min(max_images, num_img)
    rows = 2
    cols = (n + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    plt.suptitle("Test predictions: classification + localisation",
                 fontsize=18, fontweight="bold")

    for i in range(n):
        ax = axes[i]
        # Copy image to avoid modifying original
        img = images[i].copy() 
        h, w, _ = img.shape

        # Draw Box
        bx = box_preds[i]
        x1 = int(bx[0] * w)
        y1 = int(bx[1] * h)
        x2 = int(bx[2] * w)
        y2 = int(bx[3] * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # Draw Label
        lbl_idx = int(np.argmax(cls_probs[i]))
        conf = float(cls_probs[i][lbl_idx] * 100)
        lbl_name = class_names[lbl_idx]

        ax.imshow(img)
        ax.set_title(f"{lbl_name} {conf:.1f}%", fontsize=12)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def run_visual_demo(regressor, classifier, le, test_dir, tiff_dir):
    """
    Runs inference on 10 random test images to create the visualization grid.
    """
    regressor.eval()
    classifier.eval()
    
    # 1. Pick 10 Random Images
    all_files = [f for f in os.listdir(test_dir) if (f.endswith('.png'))&(f.startswith('img_'))]
    demo_files = random.sample(all_files, 10)
    
    # 2. Prepare Data Containers
    images_np = []
    box_preds = []
    cls_probs = []
    
    # Transforms
    regressor_trans = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE_BOX, Config.IMG_SIZE_BOX)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    mean_4ch = [0.485, 0.456, 0.406, 0.5]
    std_4ch  = [0.229, 0.224, 0.225, 0.5]

    with torch.no_grad():
        for fname in demo_files:
            # Load Original for display
            img_path = os.path.join(test_dir, fname)
            rgb_orig = Image.open(img_path).convert("RGB")
            images_np.append(np.array(rgb_orig))
            w, h = rgb_orig.size
            
            # A. Regressor
            regressor_input = regressor_trans(rgb_orig).unsqueeze(0).to(Config.DEVICE)
            box = regressor(regressor_input).cpu().numpy()[0] # [x1, y1, x2, y2] normalized
            box_preds.append(box)
            
            # B. CROP
            x1 = int(max(0, box[0]*w)); y1 = int(max(0, box[1]*h))
            x2 = int(min(w, box[2]*w)); y2 = int(min(h, box[3]*h))
            
            # Fallback if crop is empty
            if x2 <= x1 or y2 <= y1: x1, y1, x2, y2 = 0, 0, w, h
            rgb_crop = rgb_orig.crop((x1, y1, x2, y2))
            
            # TIFF
            t_name = fname.replace(".png", ".tif")
            t_path = os.path.join(tiff_dir, t_name)
            if os.path.exists(t_path):
                try:
                    t = tiff.imread(t_path)
                    t_norm = (t - t.min()) / (t.max() - t.min() + 1e-6)
                    tif_img = Image.fromarray((t_norm*255).astype(np.uint8)).convert("L")
                    tif_crop = tif_img.crop((x1, y1, x2, y2))
                except: tif_crop = Image.new('L', rgb_crop.size, 0)
            else: tif_crop = Image.new('L', rgb_crop.size, 0)

            # C. Classifier
            rgb_final = rgb_crop.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
            tif_final = tif_crop.resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS), Image.BILINEAR)
            
            rgb_t = TF.to_tensor(rgb_final)
            tif_t = TF.to_tensor(tif_final)
            full_t = torch.cat([rgb_t, tif_t], dim=0)
            full_t = TF.normalize(full_t, mean=mean_4ch, std=std_4ch).unsqueeze(0).to(Config.DEVICE)
            
            # Predict
            out = classifier(full_t)
            prob = F.softmax(out, dim=1).cpu().numpy()[0]
            cls_probs.append(prob)
            
    # 3. Call the plotting function
    visualize_test_predictions(images_np, cls_probs, box_preds, le.classes_, max_images=10)

In [ ]:
import umap

def visualize_feature_space(model, loader, le, device):
    """
    Extracts features from the penultimate layer of ConvNeXt and projects them using UMAP.
    Paper Insight: "Clusters of different categories... show clear boundaries."
    """
    print("\n🎨 Projecting Feature Space with UMAP...")
    model.eval()
    features_list = []
    labels_list = []
    
    # 1. Extract Embeddings
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Extracting Features"):
            imgs = imgs.to(device)
            
            # --- Manual Forward Pass to get Embeddings (ConvNeXt Specific) ---
            # We want the output BEFORE model.classifier[2] (the Linear layer)
            x = model.features(imgs)
            x = model.avgpool(x)
            x = model.classifier[0](x) # LayerNorm
            x = model.classifier[1](x) # Flatten
            # Result: (Batch_Size, 768) feature vector
            
            features_list.append(x.cpu().numpy())
            labels_list.append(lbls.numpy())

    features = np.concatenate(features_list)
    labels = np.concatenate(labels_list)
    
    # 2. Run UMAP
    # Settings from BEPH Paper: n_neighbors=50, min_dist=0.6 (or defaults 30/0.3)
    print("   Computing UMAP projection...")
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, metric='cosine', random_state=42)
    embedding = reducer.fit_transform(features)
    
    # 3. Plot
    plt.figure(figsize=(12, 10))
    
    # Create DataFrame for Seaborn
    df_plot = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'])
    df_plot['Label'] = le.inverse_transform(labels)
    
    sns.scatterplot(
        data=df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        hue='Label', 
        palette='tab10', 
        s=15, 
        alpha=0.7
    )
    
    plt.title(f"ConvNeXt Feature Space (UMAP)\nTotal Samples: {len(labels)}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

---

## MAIN EXECUTION

In [ ]:
if os.path.exists(Config.CLEAN_LABELS):
    df = pd.read_csv(Config.CLEAN_LABELS)
else:
    print("❌ Cleaned data not found.")  

In [ ]:
# 1. Train regressor
rgss, box_hist = train_regressor(df)

In [ ]:
# 2. Train Classifier
clss, le, cls_hist, best_f1, val_loader = train_classifier(df)

In [ ]:
# 3. Inference -- Final Submission
smart_inference(rgss, clss, le, Config.TEST_DIR, Config.TEST_TIFF_DIR, f1_score=best_f1)

In [ ]:
# 4. Visualize Results

# Regressor History
plot_history_box(box_hist)

# Classifier History and Confusion Matrix
plot_history_cls(cls_hist, title_suffix=" (Classifier)")
plot_confusion_matrix_model(clss, val_loader, le.classes_)

# NEW: UMAP Feature Visualization
visualize_feature_space(clss, val_loader, le, Config.DEVICE)

# Visual Demo (Grid of Images)
run_visual_demo(rgss, clss, le, Config.TEST_DIR, Config.TEST_TIFF_DIR)

---

In [ ]:
# # =========================================================================================
# # 🗑️ PRIORITY 2: THE PURGE (Loss-Based Outlier Removal)
# # =========================================================================================

# def run_the_purge(df, removal_percentile=0.05): # Remove top 5% worst images
#     print("\nSTARTING THE PURGE: Hunting for Traitors (High Loss Samples)...")
    
#     # 1. Setup Quick Hunter Model (ResNet18)
#     model = models.resnet18(weights='IMAGENET1K_V1')
#     model.fc = nn.Linear(model.fc.in_features, Config.NUM_CLASSES)
    
#     # Use 4-Channel Dataset to find artifacts in TIFF.
#     # adapting ResNet18 for 4 channels:
#     old_conv = model.conv1
#     new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
#     with torch.no_grad():
#         new_conv.weight[:, :3] = old_conv.weight
#         new_conv.weight[:, 3:4] = torch.mean(old_conv.weight, dim=1, keepdim=True)
#     model.conv1 = new_conv

#     model = model.to(Config.DEVICE)
    
#     # Dataset (No Augmentation, Cache ON)
#     ds = ClassifierDataset(df, Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TIFF_DIR, mode='val', cache_data=True)
#     loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    
#     criterion = nn.CrossEntropyLoss(reduction='none') # ⚠️ Per-sample loss
#     optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
#     # 2. Quick Training (5 Epochs is enough to spot outliers)
#     print("   Training Hunter Model (5 Epochs)...")
#     model.train()
#     for epoch in range(5):
#         for imgs, lbls in tqdm(loader, desc=f"Hunter Ep {epoch+1}", leave=False):
#             imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
#             optimizer.zero_grad()
#             out = model(imgs)
#             loss = criterion(out, lbls).mean()
#             loss.backward()
#             optimizer.step()
            
#     # 3. Calculate Loss Per Sample
#     print("   Calculating Loss for every image...")
#     model.eval()
#     losses = []
    
#     with torch.no_grad():
#         for i, (imgs, lbls) in enumerate(tqdm(loader, desc="Scanning")):
#             imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
#             out = model(imgs)
#             batch_losses = criterion(out, lbls).cpu().numpy()
#             losses.extend(batch_losses)
            
#     df['purge_loss'] = losses
    
#     # 4. Identify Traitors
#     # Sort by loss (High to Low)
#     df_sorted = df.sort_values(by='purge_loss', ascending=False)
    
#     n_remove = int(len(df) * removal_percentile)
#     traitors = df_sorted.head(n_remove)
#     survivors = df_sorted.iloc[n_remove:]
    
#     print(f"\nPURGE RESULTS:")
#     print(f"   Total Images: {len(df)}")
#     print(f"   Traitors Removed: {n_remove}")
#     print(f"   Survivors: {len(survivors)}")
#     print(f"   Loss Threshold: {traitors['purge_loss'].min():.4f}")
    
#     # Visualize Top 5 Traitors
#     print("\n   Top 5 Worst Offenders (Visual Check):")
#     plt.figure(figsize=(15, 4))
#     for i in range(5):
#         row = traitors.iloc[i]
#         img_path = os.path.join(Config.CLEAN_IMG_DIR, row['sample_index'])
#         img = Image.open(img_path)
#         plt.subplot(1, 5, i+1)
#         plt.imshow(img)
#         plt.title(f"Loss: {row['purge_loss']:.2f}\n{row['label']}")
#         plt.axis('off')
#     plt.show()
    
#     # 5. Save Cleaned CSV
#     new_csv_path = "cleaned_labels_v2.csv"
#     survivors.to_csv(new_csv_path, index=False)
#     print(f"✅ Cleaned Dataset saved to: {new_csv_path}")
    
#     return new_csv_path, traitors

# # EXECUTE PURGE
# if os.path.exists(Config.CLEAN_LABELS):
#     df_full = pd.read_csv(Config.CLEAN_LABELS)
#     PURGED_CSV_PATH, traitors = run_the_purge(df_full, removal_percentile=0.05) # Removing top 5% garbage
    
#     # UPDATE CONFIG TO USE NEW FILE
#     Config.CLEAN_LABELS = PURGED_CSV_PATH

#     print(traitors)